# ESRGAN&REMBG MODEL
**Upload/camera → automatic quality check → enhanced photo + transparent background-removed photo.**

Real-ESRGAN is unchanged. The second output uses rembg (U²-Net) with ONNX Runtime CUDA. Both use the same working input independently; background removal does not enhance the photo. Outputs retain original dimensions; the cutout is RGBA PNG.

1. Stop the previous notebook and restart its kernel to release GPU memory.
2. Select your existing working CUDA PyTorch environment (Python 3.10–3.12).
3. Run **Setup** once, then **restart the kernel** if packages were installed.
4. Run **Backend → Load models → Launch studio**. Allow the first model download to finish.

Existing Real-ESRGAN weights are reused. No API key or training required. Local processing stays on your machine; Colab uses its runtime and a temporary public Gradio URL. Keep the notebook running and allow camera permission.


## Setup · first run only

In [1]:
import sys, subprocess, importlib.util
from importlib import metadata

if sys.version_info < (3, 10):
    raise RuntimeError('Use Python 3.10 or newer in your existing CUDA PyTorch environment.')
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Select the existing notebook environment where CUDA PyTorch already works.')

def installed(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

# The GPU package is rembg[gpu], not a separate package called rembg-gpu.
requirements = {'numpy': 'numpy>=1.26,<3', 'PIL': 'Pillow>=10,<12',
                'cv2': 'opencv-python-headless==4.10.0.84', 'gradio': 'gradio==5.49.1'}
missing = [package for module, package in requirements.items()
           if importlib.util.find_spec(module) is None]
if not installed('rembg'):
    missing.append('rembg[gpu]>=2.0.61,<3')
# CPU and GPU ONNX Runtime distributions share files; keep only the GPU distribution.
changed = False
if installed('onnxruntime'):
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'onnxruntime'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--force-reinstall',
                           '--no-deps', 'onnxruntime-gpu>=1.21,<2'])
    changed = True
elif not installed('onnxruntime-gpu'):
    missing.append('onnxruntime-gpu>=1.21,<2')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
    changed = True
print('Setup complete. RESTART THE KERNEL before Backend.' if changed else
      'Setup already complete; continue with Backend.')
print('Existing PyTorch installation was left unchanged.')


Setup already complete; continue with Backend.
Existing PyTorch installation was left unchanged.


In [2]:
# %pip uninstall -y onnxruntime onnxruntime-gpu
# %pip install --no-cache-dir --force-reinstall "onnxruntime-gpu==1.26.0"

## Backend · quality gate, models, automatic memory management
The RRDBNet implementation is retained from your edited notebook. Quality measurement still uses OpenCV, but no longer rejects a valid light/dark product solely because of average brightness or low edge variance. Thresholds are conservative starting points, **not calibrated probabilities**. A noisy image can still score highly and a smooth craft can score poorly; validate on real product photos before production.

Developer policy: 24 MP upload limit; automatic working-edge cap of 512–1536 based on available CUDA memory. Larger photos are reduced internally and the report records this. Reduce-resize does not recover lost resolution detail. Models run sequentially in float32; smaller tile retries and CPU recovery handle CUDA out-of-memory errors. Tiled output can differ from full-image inference.

Sources: [Real-ESRGAN](https://github.com/xinntao/Real-ESRGAN), [rembg GPU installation](https://github.com/danielgatis/rembg#installation), [ONNX Runtime CUDA compatibility](https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html).

rembg requires a working CUDA execution provider, compatible CUDA/cuDNN libraries, and the NVIDIA driver. Importing PyTorch and ONNX Runtime library preloading help locate the installed libraries. A CUDA session check prevents silently presenting CPU-only execution as GPU execution. The rembg session is released after its turn so it does not hold GPU memory while Real-ESRGAN runs on the next upload. Session creation adds some per-image overhead.


In [3]:
import gc
import hashlib
import importlib.util
import json
import math
import shutil
import threading
import time
import urllib.request
import uuid
import zipfile
from pathlib import Path

import cv2
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from PIL import Image, ImageOps, ImageEnhance, ImageFilter

ROOT = Path.cwd() / 'karigarkart_upscaler'
WEIGHT_DIR = ROOT / 'weights'
OUTPUTS = ROOT / 'outputs'
WEIGHT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
JOB_LOCK = threading.Lock()  # shared by UI and future API callers
MODELS = {}
MODEL_ERRORS = {}
MAX_INPUT_PIXELS = 24_000_000



class DenseResidual(nn.Module):
    def __init__(self):
        super().__init__()
        for i in range(5):
            setattr(self, f'conv{i+1}', nn.Conv2d(64 + 32*i, 32 if i < 4 else 64, 3, padding=1))

    def forward(self, x):
        features = [x]
        for i in range(1, 5):
            features.append(F.leaky_relu(getattr(self, f'conv{i}')(torch.cat(features, dim=1)), 0.2))
        return x + 0.2 * self.conv5(torch.cat(features, dim=1))

class TripleResidual(nn.Module):
    def __init__(self):
        super().__init__()
        self.rdb1, self.rdb2, self.rdb3 = DenseResidual(), DenseResidual(), DenseResidual()

    def forward(self, x):
        return x + 0.2 * self.rdb3(self.rdb2(self.rdb1(x)))

class RRDBNetX4(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_first = nn.Conv2d(3, 64, 3, padding=1)
        self.body = nn.Sequential(*[TripleResidual() for _ in range(23)])
        self.conv_body = nn.Conv2d(64, 64, 3, padding=1)
        self.conv_up1 = nn.Conv2d(64, 64, 3, padding=1)
        self.conv_up2 = nn.Conv2d(64, 64, 3, padding=1)
        self.conv_hr = nn.Conv2d(64, 64, 3, padding=1)
        self.conv_last = nn.Conv2d(64, 3, 3, padding=1)

    def forward(self, x):
        x = self.conv_first(x)
        x = x + self.conv_body(self.body(x))
        for layer in (self.conv_up1, self.conv_up2):
            x = F.leaky_relu(layer(F.interpolate(x, scale_factor=2, mode='nearest')), 0.2)
        return self.conv_last(F.leaky_relu(self.conv_hr(x), 0.2))


def normalize_image(image):
    if image is None:
        raise ValueError('Upload an image or take a photo first.')
    if not isinstance(image, Image.Image):
        raise TypeError('Expected a PIL.Image.Image.')
    if image.width * image.height > MAX_INPUT_PIXELS:
        raise ValueError('Photo is too large. Upload a photo below 24 megapixels.')
    image = ImageOps.exif_transpose(image)
    if image.mode in ('RGBA', 'LA', 'P') and ('A' in image.getbands() or 'transparency' in image.info):
        rgba = image.convert('RGBA')
        image = Image.alpha_composite(Image.new('RGBA', rgba.size, 'white'), rgba)
    return image.convert('RGB')


def check_image_quality(image):
    """Conservative heuristic, not a learned quality probability or product detector."""
    image = normalize_image(image)
    preview = image.copy()
    preview.thumbnail((768, 768), Image.Resampling.LANCZOS)
    gray = cv2.cvtColor(np.asarray(preview), cv2.COLOR_RGB2GRAY)
    # Mild smoothing reduces noise masquerading as sharp edges.
    smooth = cv2.GaussianBlur(gray, (3, 3), 0)
    sharpness = float(cv2.Laplacian(smooth, cv2.CV_64F).var())
    brightness, contrast = float(gray.mean()), float(gray.std())
    p01, p99 = np.percentile(gray, [1, 99]).astype(float)
    shortest = min(image.size)
    sharp_score = float(np.interp(sharpness, [0, 2, 8, 25, 80], [0, 15, 40, 75, 100]))
    light_score = float(np.interp(brightness, [0, 20, 65, 200, 240, 255], [0, 25, 100, 100, 25, 0]))
    size_score = float(np.interp(shortest, [0, 160, 320, 720], [0, 15, 60, 100]))
    contrast_score = float(np.interp(contrast, [0, 5, 15, 40], [0, 10, 55, 100]))
    score = int(round(.45*sharp_score + .20*light_score + .25*size_score + .10*contrast_score))
    reject = []
    warnings = []
    if shortest < 160:
        reject.append('Photo resolution is too small. Upload the original camera photo.')
    if p99 < 18:
        reject.append('Almost no visible detail. Take the photo in better light.')
    elif p01 > 242:
        reject.append('Photo is almost completely white. Reduce exposure and retake.')
    elif contrast < 1.5:
        reject.append('No useful visible detail. Focus on the product and retake.')
    elif sharpness < 1.2 and contrast < 12:
        reject.append('Very little detail and low contrast. Hold the camera steady and refocus.')
    if sharpness < 8:
        warnings.append('Possible blur or a smooth/plain subject; check product details.')
    if brightness < 55:
        warnings.append('Dark photo; better lighting may help.')
    if brightness > 225:
        warnings.append('Bright photo; check highlights on the product.')
    if shortest < 512:
        warnings.append('Low resolution; an original camera photo is preferable.')
    # Do not hard reject by average brightness or sharpness alone:
    # white backgrounds, black products and untextured crafts can be valid.
    status = 'Retake' if reject else ('Good' if score >= 75 else 'Bad Quality')
    return dict(score=score, status=status, accepted=not reject,
                message=('Poor image — upload again. ' + ' '.join(reject)) if reject else
                ('Photo accepted. Creating two options.' if status == 'Good' else
                 'Photo accepted with quality concerns. We will enhance it and remove the background.'),
                reasons=reject, warnings=warnings,
                metrics=dict(sharpness=round(sharpness, 2), brightness=round(brightness, 1),
                             contrast=round(contrast, 1), resolution=list(image.size),
                             black_percent=round(float((gray <= 5).mean()*100), 1),
                             white_percent=round(float((gray >= 250).mean()*100), 1)),
                score_type='uncalibrated heuristic; not confidence probability')


def download_once(url, destination, minimum_bytes=1000):
    destination = Path(destination)
    if destination.exists():
        if destination.stat().st_size < minimum_bytes:
            raise RuntimeError(f'Incomplete cached download: {destination}. Replace this file from {url}.')
        return destination
    temporary = destination.with_name(destination.name + '.' + uuid.uuid4().hex + '.part')
    try:
        request = urllib.request.Request(url, headers={'User-Agent': 'KARIGARKART'})
        with urllib.request.urlopen(request, timeout=120) as response, temporary.open('wb') as target:
            shutil.copyfileobj(response, target)
        if temporary.stat().st_size < minimum_bytes:
            raise RuntimeError('Downloaded file is unexpectedly small.')
        temporary.replace(destination)
    except Exception as error:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(f'Download failed. Download {url} manually to {destination.resolve()}') from error
    return destination


def _new_rembg_session():
    import onnxruntime as ort
    from rembg import new_session
    if not torch.cuda.is_available():
        raise RuntimeError('rembg GPU needs CUDA PyTorch in this kernel. Check torch.cuda.is_available().')
    if hasattr(ort, 'preload_dlls'):
        ort.preload_dlls()
    if 'CUDAExecutionProvider' not in ort.get_available_providers():
        raise RuntimeError('CUDAExecutionProvider unavailable. Run Setup and restart the kernel.')
    session = new_session('u2net', providers=[
        ('CUDAExecutionProvider', {'device_id': torch.cuda.current_device(),
                                   'arena_extend_strategy': 'kSameAsRequested',
                                   'cudnn_conv_algo_search': 'HEURISTIC',
                                   'cudnn_conv_use_max_workspace': '0'}),
        'CPUExecutionProvider',
    ])
    actual = session.inner_session.get_providers()
    if 'CUDAExecutionProvider' not in actual:
        del session
        gc.collect()
        raise RuntimeError('rembg CUDA session failed to initialize. Check matching CUDA/cuDNN libraries '
                           'for onnxruntime-gpu and PyTorch; inspect the ONNX Runtime error above.')
    session.inner_session.disable_fallback()
    return session


def run_background_removal(image, original, progress=None):
    from rembg import remove
    session = None
    try:
        if progress:
            progress(0.05)
        session = _new_rembg_session()
        providers = session.inner_session.get_providers()
        # Predict a mask at working size; retain original product RGB and dimensions.
        mask = remove(image, session=session, only_mask=True, alpha_matting=False)
        mask = mask.convert('L').resize(original.size, Image.Resampling.LANCZOS)
        result = original.convert('RGBA')
        result.putalpha(mask)
        if progress:
            progress(1.0)
        return result, dict(device='cuda', providers=providers, model='u2net',
                            alpha=True, input='original working image',
                            mask_size=list(image.size))
    finally:
        if session is not None:
            del session
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def load_models():
    """Keep ESRGAN on CPU between jobs; validate then release rembg's CUDA session."""
    with JOB_LOCK:
        MODEL_ERRORS.clear()
        for name in ('Real-ESRGAN', 'rembg'):
            if name in MODELS:
                continue
            try:
                if name == 'Real-ESRGAN':
                    path = download_once('https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
                                         WEIGHT_DIR / 'RealESRGAN_x4plus.pth', 1_000_000)
                    network = RRDBNetX4()
                    checkpoint = torch.load(path, map_location='cpu', weights_only=True)
                    state = checkpoint.get('params_ema', checkpoint.get('params', checkpoint))
                    network.load_state_dict(state, strict=True)
                    network.eval().float().cpu()
                    with torch.inference_mode():
                        probe = network(torch.zeros(1, 3, 16, 16))
                        if probe.shape != (1, 3, 64, 64) or not torch.isfinite(probe).all().item():
                            raise RuntimeError('Real model smoke test failed.')
                    MODELS[name] = network
                    del checkpoint, state, probe
                else:
                    sample = Image.new('RGB', (64, 64), 'white')
                    probe, execution = run_background_removal(sample, sample)
                    if probe.mode != 'RGBA' or probe.size != sample.size:
                        raise RuntimeError('rembg output smoke test failed.')
                    MODELS[name] = {'model': 'u2net', 'providers': execution['providers']}
                    del probe
                print(name, 'ready; inference smoke test passed.')
            except Exception as error:
                MODEL_ERRORS[name] = str(error)
                print(name, 'not ready:', error)
        return {'ready': list(MODELS), 'errors': dict(MODEL_ERRORS)}


def auto_settings():
    free_gb = 0.0
    if DEVICE.type == 'cuda':
        free_gb = torch.cuda.mem_get_info(DEVICE)[0] / 1024**3
    return dict(device=str(DEVICE),
                max_edge=1536 if free_gb >= 4 else (1024 if free_gb >= 2 else 768 if free_gb > 0 else 512),
                tile=192 if free_gb >= 6 else 128 if free_gb >= 3 else 64,
                free_gpu_gb=round(free_gb, 2))


@torch.inference_mode()
def _tile_same_size(network, image, tile, device, progress=None):
    """Internal x4 prediction, then halo-aware downsampling to 1x per tile.
    Stores only the final 1x canvas on CPU, avoiding a huge 4x canvas.
    Original tile padding is retained for consistent Real-ESRGAN output.
    """
    pixels = np.asarray(image, dtype=np.uint8)
    h, w = pixels.shape[:2]
    output = np.empty_like(pixels)
    total = math.ceil(h/tile) * math.ceil(w/tile)
    done = 0
    for y in range(0, h, tile):
        for x in range(0, w, tile):
            r, b = min(w, x+tile), min(h, y+tile)
            l, t = max(0, x-32), max(0, y-32)
            rp, bp = min(w, r+32), min(h, b+32)
            patch = np.ascontiguousarray(pixels[t:bp, l:rp])
            tensor = torch.from_numpy(patch).permute(2, 0, 1).unsqueeze(0).to(device=device, dtype=torch.float32)/255
            ph, pw = tensor.shape[-2:]
            dh, dw = (-ph)%8, (-pw)%8
            if dh or dw:
                tensor = F.pad(tensor, (0, dw, 0, dh), mode='reflect' if ph>dh and pw>dw else 'replicate')
            pred = network(tensor)[0, :, :ph*4, :pw*4]
            if not torch.isfinite(pred).all().item():
                raise RuntimeError('Non-finite model output; no image was saved.')
            array = (pred.clamp(0,1).permute(1,2,0).cpu().numpy()*255).round().astype(np.uint8)
            restored = np.asarray(Image.fromarray(array).resize((pw,ph), Image.Resampling.LANCZOS))
            output[y:b, x:r] = restored[y-t:b-t, x-l:r-l]
            del tensor, pred, array, restored
            done += 1
            if progress:
                progress(done/total)
    return Image.fromarray(output)


def run_network(name, image, settings, progress=None):
    network = MODELS[name]
    candidates = sorted(set([settings['tile'], 96, 64, 32]) & set(range(32, settings['tile']+1)), reverse=True)
    attempts = []
    # Retry smaller CUDA tiles; use CPU as final automatic recovery.
    for device in ([DEVICE, torch.device('cpu')] if DEVICE.type == 'cuda' else [DEVICE]):
        for tile in (candidates if device.type == 'cuda' else [64]):
            failed_oom = False
            try:
                network.to(device=device, dtype=torch.float32)
                result = _tile_same_size(network, image, tile, device, progress)
                return result, dict(device=str(device), tile=tile, retries=attempts)
            except torch.cuda.OutOfMemoryError:
                if device.type != 'cuda':
                    raise
                failed_oom = True
            finally:
                network.cpu()
                if DEVICE.type == 'cuda' and not failed_oom:
                    torch.cuda.empty_cache()
            # Outside exception context so traceback no longer retains tile tensors.
            if failed_oom:
                attempts.append({'device': str(device), 'tile': tile, 'reason': 'CUDA memory'})
                gc.collect()
                torch.cuda.empty_cache()
    raise RuntimeError('Unable to process this photo on the available hardware.')


def process_image(image, progress=None):
    """UI-independent API boundary. PIL input -> JSON-serializable report and local file paths.
    Always rechecks quality. Never trusts a browser-supplied score.
    """
    original = normalize_image(image)
    quality = check_image_quality(original)
    if not quality['accepted']:
        return dict(status='retake', quality=quality, variants={}, errors={})
    with JOB_LOCK:
        start = time.perf_counter()
        settings = auto_settings()
        working = original.copy()
        working.thumbnail((settings['max_edge'], settings['max_edge']), Image.Resampling.LANCZOS)
        job_id = uuid.uuid4().hex
        folder = OUTPUTS / job_id
        folder.mkdir()
        original.save(folder / 'original.png')


        variants, errors = {}, {}

        for index, name in enumerate(('Real-ESRGAN', 'rembg')):
            if name not in MODELS:
                errors[name] = MODEL_ERRORS.get(
                    name, 'Model not loaded. Run the model setup cell.'
                )
                continue

            try:
                def report_fraction(fraction):
                    if progress:
                        progress(
                            (index + fraction) / 2,
                            f'Creating option {index + 1} of 2'
                        )

                # OPTION 1: Original → Real-ESRGAN
                if name == 'Real-ESRGAN':
                    result, execution = run_network(
                        name, working, settings, report_fraction
                    )

                    result = result.resize(
                        original.size, Image.Resampling.LANCZOS
                    )
                    result = ImageEnhance.Contrast(result).enhance(1.03)
                    result = result.filter(
                        ImageFilter.UnsharpMask(
                            radius=1, percent=35, threshold=4
                        )
                    )
                    filename = 'realesrgan.png'

                # OPTION 2: rembg → Real-ESRGAN
                else:
                    cutout, rembg_execution = run_background_removal(
                        working,
                        working,
                        lambda f: report_fraction(f * 0.4)
                    )

                    alpha = cutout.getchannel('A')

                    # ESRGAN requires RGB, not RGBA
                    cutout_rgb = Image.new('RGB', cutout.size, 'white')
                    cutout_rgb.paste(
                        cutout.convert('RGB'),
                        mask=alpha
                    )

                    result, esr_execution = run_network(
                        'Real-ESRGAN',
                        cutout_rgb,
                        settings,
                        lambda f: report_fraction(0.4 + f * 0.6)
                    )

                    result = result.resize(
                        original.size, Image.Resampling.LANCZOS
                    )
                    result = ImageEnhance.Contrast(result).enhance(1.03)
                    result = result.filter(
                        ImageFilter.UnsharpMask(
                            radius=1, percent=35, threshold=4
                        )
                    )

                    # Transparency dobara enhanced image par lagao
                    alpha = alpha.resize(
                        original.size, Image.Resampling.LANCZOS
                    )
                    result = result.convert('RGBA')
                    result.putalpha(alpha)

                    execution = {
                        'pipeline': ['rembg', 'Real-ESRGAN'],
                        'rembg': rembg_execution,
                        'enhancement': esr_execution
                    }
                    filename = 'rembg_then_realesrgan.png'

                path = folder / filename
                result.save(path)

                variants[name] = {
                    'path': str(path.resolve()),
                    'width': result.width,
                    'height': result.height,
                    'model': name,
                    'execution': execution
                }

            except Exception as error:
                errors[name] = str(error)

    
         # A failed model never impersonates the other.
        report = dict(job_id=job_id, status='complete' if len(variants)==2 else 'partial' if variants else 'failed',
                      quality=quality, variants=variants, errors=errors,
                      original_size=list(original.size), processed_size=list(working.size),
                      resized_for_processing=working.size != original.size,
                      settings=settings, seconds=round(time.perf_counter()-start, 2))
        manifest = folder / 'report.json'
        manifest.write_text(json.dumps(report, indent=2), encoding='utf-8')
        archive = folder / 'all_photos.zip'
        with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
            for path in [folder/'original.png', manifest] + [Path(v['path']) for v in variants.values()]:
                bundle.write(path, arcname=path.name)
        report['report_path'] = str(manifest.resolve())
        report['download_all'] = str(archive.resolve())
        return report


## Load models · once per kernel session
Real-ESRGAN loads the existing checkpoint and checks inference. rembg downloads U²-Net on first use, verifies a real CUDA session with a small inference, then releases it. A ready marker is retained; the session is recreated per job to free GPU memory between uploads. Fix any displayed download/CUDA error and rerun this cell; successful models are reused.


In [4]:
setup_report = load_models()
print('Hardware:', DEVICE)
print(setup_report)


Real-ESRGAN ready; inference smoke test passed.


c:\SIH\VEDANT\vedantsih_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Skip loading CUDA and cuDNN DLLs since torch is imported.
rembg ready; inference smoke test passed.
Hardware: cuda
{'ready': ['Real-ESRGAN', 'rembg'], 'errors': {}}


## Launch studio
Allow the job to finish before choosing another photo. If you edit/relaunch this cell and see an event-loop error, restart the kernel and rerun Backend → Load models → Launch studio. A failed model is reported explicitly; the successful variant remains downloadable.

In [5]:
import gradio as gr

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if 'demo' in globals():
    demo.close()


def automatic_upload(image, progress=gr.Progress()):
    # Order: photo, quality_message, esr_preview,
    #        rembg_preview, details, retry
    if image is None:
        yield (
            gr.update(interactive=True),
            "Upload a product photo to begin.",
            None, None, {},
            gr.update(interactive=False),
        )
        return

    yield (
        gr.update(interactive=False),
        "Checking your photo...",
        None, None, {},
        gr.update(interactive=False),
    )

    try:
        quality = check_image_quality(image)
        summary = (
            f"**Photo quality: {quality['score']}/100"
            f" - {quality['status']}**\n\n{quality['message']}"
        )

        if not quality["accepted"]:
            yield (
                gr.update(interactive=True),
                summary, None, None, quality,
                gr.update(interactive=False),
            )
            return

        yield (
            gr.update(interactive=False),
            summary, None, None, quality,
            gr.update(interactive=False),
        )

        report = process_image(
            image,
            progress=lambda fraction, desc: progress(
                fraction, desc=desc
            ),
        )

        variants = report.get("variants", {})
        a = variants.get("Real-ESRGAN", {}).get("path")
        b = variants.get("rembg", {}).get("path")

        if report["status"] == "complete":
            summary += "\n\n**Your two photos are ready. Choose one.**"
        elif report["status"] == "partial":
            summary += "\n\n**One photo is ready.** See technical details."
        else:
            summary += "\n\n**Processing could not finish.** See technical details."

        yield (
            gr.update(interactive=True),
            summary, a, b, report,
            gr.update(interactive=True),
        )

    except Exception as error:
        import traceback
        traceback.print_exc()

        yield (
            gr.update(interactive=True),
            "Unable to process this photo. See technical details.",
            None, None, {"error": str(error)},
            gr.update(interactive=True),
        )


with gr.Blocks(title='KARIGARKART Photo Studio') as demo:
    gr.Markdown('# KARIGARKART - Photo Studio\nUpload or take a product photo. We check it and create an enhanced photo and a background-removed photo automatically.')
    photo = gr.Image(sources=['upload', 'webcam'], type='pil', label='Your product photo')
    quality_message = gr.Markdown('Upload a product photo to begin.')
    with gr.Row():
        with gr.Column():
            gr.Markdown('### Option 1 - Real-ESRGAN')
            esr_preview = gr.Image(label='Enhanced photo 1', interactive=False)
            # esr_download = gr.File(label='Download photo 1', interactive=False)
        with gr.Column():
            gr.Markdown('### Option 2 - Background removed')
            rembg_preview = gr.Image(label='Background removed (transparent PNG)', image_mode='RGBA', format='png', interactive=False)
            # rembg_download = gr.File(label='Download photo 2', interactive=False)
    # all_downloads = gr.File(label='Download all photos + report', interactive=False)
    retry = gr.Button('Try this photo again', interactive=False)
    with gr.Accordion('Technical details', open=False):
        details = gr.JSON(label='Quality and processing report')
    gr.Markdown('Compare fine patterns, colours and lettering with your original before listing the product.')
    ui_outputs = [photo, quality_message, esr_preview, rembg_preview, details, retry]
    photo.input(automatic_upload, inputs=photo, outputs=ui_outputs,
                trigger_mode='always_last', concurrency_id='studio', concurrency_limit=1)
    retry.click(automatic_upload, inputs=photo, outputs=ui_outputs,
                concurrency_id='studio', concurrency_limit=1)

demo.queue(max_size=8).launch(share=IN_COLAB, inline=False, show_error=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Future React / mobile integration
`check_image_quality(PIL_image)` returns a JSON-ready score/reasons/status.
`process_image(PIL_image)` checks quality again and returns a JSON-ready result:

- `status`: `retake`, `complete`, `partial`, or `failed`.
- `quality`: score, accepted flag, reasons and metrics.
- `variants`: separate `Real-ESRGAN` / `rembg` records containing saved PNG paths and execution details.
- `job_id`: unique output directory; requests do not overwrite each other.
- `original_size`, `processed_size`, `resized_for_processing`: transparent sizing policy.
- `errors`: per-model errors; no hidden substitution.

For FastAPI: load the module/models once at application startup; decode the uploaded bytes to PIL, call `process_image` in a worker thread and map returned paths to authenticated image/download endpoints. A local disk path is **not** a browser URL. Keep GPU inference in one worker/queue (the Python lock serializes threads within one process only). Add authentication, per-user job ownership, file-size limits, cleanup and a durable job queue before multi-user deployment. The current Gradio UI is a local prototype, not that deployment.

The optional cell below exports the backend functions into an importable `.py` module beside the notebook. It performs no model downloads on import; call `load_models()` explicitly. Set `NOTEBOOK_PATH` to the filename you actually saved.


In [6]:
# EXPORT_BACKEND = False  # Change to True only when you want the .py file.
# if EXPORT_BACKEND:
#     import json
#     from pathlib import Path
#     NOTEBOOK_PATH = Path('ESRGAN&REMBG_MODEL.ipynb')  # Your saved notebook path
#     notebook = json.loads(NOTEBOOK_PATH.read_text(encoding='utf-8'))
#     blocks = [''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
#               for cell in notebook['cells']
#               if 'backend-core' in cell.get('metadata', {}).get('tags', [])]
#     if not blocks:
#         raise RuntimeError('This is not the ESRGAN and rembg notebook.')
#     destination = NOTEBOOK_PATH.with_name('karigarkart_engine.py')
#     destination.write_text('\n\n'.join(blocks), encoding='utf-8')
#     print('Exported:', destination.resolve())


## Validation and limits
Code cells are syntax-checked; orchestration is tested with substituted model outputs. Real pretrained inference, CUDA memory behavior and browser-camera interaction have not been executed in this authoring environment. **Load models** performs real inference checks on your laptop.

PNG outputs and reports remain in `karigarkart_upscaler/outputs/<job_id>`. Input alpha is composited on white, as before. The rembg output adds a predicted transparency mask to the original RGB; it does not receive the enhancement output. No sharpening or contrast adjustment is applied to its alpha channel. Check fine threads, holes, glass and product edges before listing.

Quality thresholds, Real-ESRGAN architecture/weights, automatic sizing, tiled enhancement, gentle finishing, retries, per-job folders, reports, ZIP packaging and upload/camera events are retained. Background removal failure leaves the enhanced photo available, and vice versa. No new download controls were added. There is no second enhancement model.
